In [1]:
from agents import manager, user_proxy, gpt_config
import autogen
import copy
import json
import re
from collections import OrderedDict

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

In [2]:
def extract_json_block(text: str) -> str:
    """Return the first ```json ... ``` block from text, unchanged."""
    m = re.search(r"```json.*?```", text, re.DOTALL | re.IGNORECASE)
    return m.group(0) if m else ""

def build_available_tools(tools: list[dict]) -> dict:
    """
    Convert an OpenAI-style tools list into a compact, JSON-serialisable mapping:
    {
      "tool_name": {
        "description": "...",
        "parameters": {
          "required": ["arg1", "arg2"],
          "optional": ["arg3", ...],
          "schema": { ... original JSON schema for parameters ... }
        }
      },
      ...
    }
    """
    out = OrderedDict()
    for t in tools:
        if t.get("type") != "function":
            continue
        fn = t.get("function", {})
        name = fn.get("name")
        if not name:
            continue

        params = fn.get("parameters", {}) or {}
        props = params.get("properties", {}) or {}
        required = params.get("required", []) or []

        out[name] = {
            "description": fn.get("description", ""),
            "parameters": {
                "required": list(required),
                "optional": [k for k in props.keys() if k not in required],
                "schema": params,  # keep the full JSON schema for the critic to validate types
            },
        }
    return out

tools = manager.llm_config['tools']
available_tools_dict = build_available_tools(tools)
available_tools_str = json.dumps(available_tools_dict, indent=2, ensure_ascii=False)

In [3]:
EVALUATOR_SYSTEM_PROMPT = f"""
tool evaluator. Given a user provided task, you need to evaluate the chain of tools generated by an agent. Here are the available tools to the agent (name → description, parameters):
{available_tools_str}

Instructions:
1. First, check whether the overall chain makes sense to address the task, whether any important steps are missing, and whether the tools are used in a correct order.
2. Then, check each step:
    - Verify the tool exists. Make sure the tool name is correct.
    - Verify the tool is appropriate for the step. Make sure the tool usage fits the tool description.
    - Verify the provided arguments to the tool. Make sure they align with the task description provided by the user and the tool's parameter schema.

Output four scores (ranged 1-5) for each of the above checks, with 1 being the worst and 5 being the best:
    - Overall chain correctness (1-5)
    - Tool existence correctness (1-5)
    - Tool appropriateness correctness (1-5)
    - Argument correctness (1-5)
"""

# build evaluator
tool_evaluator = autogen.AssistantAgent(
    name="tool_evaluator",
    system_message=EVALUATOR_SYSTEM_PROMPT,
    llm_config=gpt_config,
)

In [4]:
prefix = '''Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:'''

In [5]:
# Evaluation begins
# Task 1: Linear Regression on Leaf Morphology
task = '''
You have a CSV file at ./data/leaf_morphology.csv containing 50 records with columns: file_name, leaf_diameter, and leaf_weight.
First, clean the data by removing any records with missing values. 
Next, fit a linear regression with leaf_diameter as x and leaf_weight as y. 
Report the equation of the regression line, the R^2 value, and generate a plot of the regression at ./results/leaf_length_vs_width.png.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a CSV file at ./data/leaf_morphology.csv containing 50 records with columns: file_name, leaf_diameter, and leaf_weight.
First, clean the data by removing any records with missing values. 
Next, fit a linear regression with leaf_diameter as x and leaf_weight as y. 
Report the equation of the regression line, the R^2 value, and generate a plot of the regression at ./results/leaf_length_vs_width.png.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.coding",
        "parameters": {
            "message": "Clean the data by removing any records with missing values.",
            "file_path": "./data/leaf_morphology.csv"
        }
    },
    {
        "recipient_name": "funct

In [6]:
# Evaluate Task 1
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to tool_evaluator):

Here is the task provided by the user: 
You have a CSV file at ./data/leaf_morphology.csv containing 50 records with columns: file_name, leaf_diameter, and leaf_weight.
First, clean the data by removing any records with missing values. 
Next, fit a linear regression with leaf_diameter as x and leaf_weight as y. 
Report the equation of the regression line, the R^2 value, and generate a plot of the regression at ./results/leaf_length_vs_width.png.

Here is the chain of tools generated by the agent:
```json
[
    {
        "recipient_name": "functions.coding",
        "parameters": {
            "message": "Clean the data by removing any records with missing values.",
            "file_path": "./data/leaf_morphology.csv"
        }
    },
    {
        "recipient_name": "functions.coding",
        "parameters": {
            "message": "Fit a linear regression with leaf_diameter as x and leaf_weight as y. Report the equation of the regression line and the R^2 va

In [7]:
# Task 2: Merging Experiments and ANOVA Analysis
task = '''
You have several CSV files in ./data/exp_results/ representing different experimental replicates. 
Merge these files on the column 'sample_id'. 
After merging, perform an ANOVA test on the column 'chlorophyll_content' across different treatment groups. 
Output the ANOVA summary table as ./results/anova_chlorophyll.csv.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 2
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have several CSV files in ./data/exp_results/ representing different experimental replicates. 
Merge these files on the column 'sample_id'. 
After merging, perform an ANOVA test on the column 'chlorophyll_content' across different treatment groups. 
Output the ANOVA summary table as ./results/anova_chlorophyll.csv.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.coding",
        "parameters": {
            "message": "Merge several CSV files located in ./data/exp_results/ on the column 'sample_id'. Save the merged file as ./data/merged_results.csv."
        }
    },
    {
        "recipient_name": "functions.perform_anova",
        "parameters": {
            "data_path": "

In [8]:
# Task 3: Area Measurement in Flower Images
task = '''
You have a folder of images at ./images/flower_plants/ and a metadata CSV at ./data/flower_metadata.csv with columns: file_name and bloom_status. 
For each image, measure its area. 
Merge these area measurements with the metadata, and then create a boxplot comparing area distributions for different bloom_status values. 
Save the boxplot as ./results/flower_area_boxplot.png.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 3
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a folder of images at ./images/flower_plants/ and a metadata CSV at ./data/flower_metadata.csv with columns: file_name and bloom_status. 
For each image, measure its area. 
Merge these area measurements with the metadata, and then create a boxplot comparing area distributions for different bloom_status values. 
Save the boxplot as ./results/flower_area_boxplot.png.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.get_model_zoo",
        "parameters": {}
    },
    {
        "recipient_name": "functions.infer_instance_segmentation",
        "parameters": {
            "file_path": "./data/flower_metadata.csv",
            "checkpoint": "instance_segmentation_checkpoint_i

In [9]:
# Task 4: Wheat Spike Phenotype Extraction
task = '''
Using the metadata file './data/wheat_metadata.json' (which contains keys: file_name, variety, and acquisition_date), extract the phenotypes {spike count, spike length} for each wheat image.
Merge these phenotypes with themetadata and save the combined data to './results/wheat_phenotypes.csv'.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 4
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
Using the metadata file './data/wheat_metadata.json' (which contains keys: file_name, variety, and acquisition_date), extract the phenotypes {spike count, spike length} for each wheat image.
Merge these phenotypes with themetadata and save the combined data to './results/wheat_phenotypes.csv'.

--------------------------------------------------------------------------------
manager (to Admin):

```json
{
  "tool_uses": [
    {
      "recipient_name": "functions.get_model_zoo",
      "parameters": {}
    },
    {
      "recipient_name": "functions.infer_instance_segmentation",
      "parameters": {
        "file_path": "./data/wheat_metadata.json",
        "checkpoint": "wheat_spike_segmentation_checkpoint",
        "output_dir": "./results"
      }
    },
    {
      "recipient_name"

In [10]:
# Task 5: Wheat Variety Spike Count Analysis
task = '''
Given './results/wheat_phenotypes.csv', perform a statistical analysis to test whether there is a significant difference in spike count between different wheat varieties. 
Use an appropriate statistical test and output the test statistics and p-value to a text file.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 5
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
Given './results/wheat_phenotypes.csv', perform a statistical analysis to test whether there is a significant difference in spike count between different wheat varieties. 
Use an appropriate statistical test and output the test statistics and p-value to a text file.

--------------------------------------------------------------------------------
manager (to Admin):

```json
{
  "tool_uses": [
    {
      "recipient_name": "functions.perform_anova",
      "parameters": {
        "data_path": "./results/wheat_phenotypes.csv",
        "descriptor": "spike_count",
        "within_subject_factor": "variety",
        "between_subjects_factor": "variety",
        "subject_id": "subject_id",
        "save_path": "./results/anova_results.csv"
      }
    },
    {
      "recipient_name": "fun

In [11]:
# Task 6: Wheat Growth Phenotyping
task = '''
You have a set of wheat images with a metadata file './data/wheat_metadata.csv' (with columns: file_name, plant_id, date, treatment). 
Compute leaf count and leaf area for each image. 
Merge the computed phenotypes with the metadata and save to './results/wheat_growth_phenotypes.csv', then generate comparative plots of leaf count and area by treatment group. 
Summarise these plots for different treatments.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 6
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a set of wheat images with a metadata file './data/wheat_metadata.csv' (with columns: file_name, plant_id, date, treatment). 
Compute leaf count and leaf area for each image. 
Merge the computed phenotypes with the metadata and save to './results/wheat_growth_phenotypes.csv', then generate comparative plots of leaf count and area by treatment group. 
Summarise these plots for different treatments.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.get_model_zoo",
        "parameters": {}
    },
    {
        "recipient_name": "functions.infer_instance_segmentation",
        "parameters": {
            "file_path": "./data/wheat_metadata.csv",
            "checkpoint": "yo

In [12]:
# Task 7: Stress Prediction
task = '''
Predict stress levels from images listed in the metadata file './data/stress_metadata.csv' (with columns: file_name, manual_stress_level).
Compare these predictions against manually scored stress levels provided in the metadata using confusion matrix. Plot the confusion matrix and save it as './results/stress_confusion_matrix.png'.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 7
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
Predict stress levels from images listed in the metadata file './data/stress_metadata.csv' (with columns: file_name, manual_stress_level).
Compare these predictions against manually scored stress levels provided in the metadata using confusion matrix. Plot the confusion matrix and save it as './results/stress_confusion_matrix.png'.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.get_model_zoo",
        "parameters": {}
    },
    {
        "recipient_name": "functions.infer_image_classification",
        "parameters": {
            "file_path": "./data/stress_metadata.csv",
            "checkpoint": "suitable_checkpoint_identifier",
            "output_dir": "./results"
       

In [13]:
# Task 8: Maximum Leaf Count Extraction
task = '''
Given './data/arabidopsis_phenotypes.csv', determine which ecotype has the maximum leaf count at day 26. 
Generate a bar plot showing the maximum leaf count for each ecotype at day 26.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 8
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
Given './data/arabidopsis_phenotypes.csv', determine which ecotype has the maximum leaf count at day 26. 
Generate a bar plot showing the maximum leaf count for each ecotype at day 26.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.query_csv",
        "parameters": {
            "message": "Which ecotype has the maximum leaf count at day 26?",
            "file_path": "./data/arabidopsis_phenotypes.csv"
        }
    },
    {
        "recipient_name": "functions.plot_from_csv",
        "parameters": {
            "message": "Generate a bar plot showing the maximum leaf count for each ecotype at day 26.",
            "file_path": "./data/arabidopsis_phenotypes.csv",
           

In [14]:
# Task 9: Extracting and Summarising Field Observations
task = '''
You have a file at ./data/field_observations.csv containing plant phenotyping observations with various attributes. 
Extract all records where the attribute'disease_symptom' is true, and generate a summary plot counting the frequency of each symptom type.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 9
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a file at ./data/field_observations.csv containing plant phenotyping observations with various attributes. 
Extract all records where the attribute'disease_symptom' is true, and generate a summary plot counting the frequency of each symptom type.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.compute_from_csv",
        "parameters": {
            "message": "Extract all records where the attribute 'disease_symptom' is true.",
            "file_path": "./data/field_observations.csv",
            "save_path": "./data/disease_symptoms.csv"
        }
    },
    {
        "recipient_name": "functions.plot_from_csv",
        "parameters": {
            "message": "Plot the 

In [15]:
# Task 10: Maize Root Phenotyping and Yield Correlation
task = '''
Given a dataset of maize root images stored in './data/maize_roots/' and metadata in './data/maize_metadata.csv' (which includes file_name, plant_id, and yield). 
Extract the total root length for each image. 
Find the correlation coefficient between the root length and yield.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 10
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
Given a dataset of maize root images stored in './data/maize_roots/' and metadata in './data/maize_metadata.csv' (which includes file_name, plant_id, and yield). 
Extract the total root length for each image. 
Find the correlation coefficient between the root length and yield.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.get_model_zoo",
        "parameters": {}
    },
    {
        "recipient_name": "functions.infer_instance_segmentation",
        "parameters": {
            "file_path": "./data/maize_metadata.csv",
            "checkpoint": "maize-root-instance-segmentation-checkpoint",
            "output_dir": "./results/instance_segmentation"
        }
    },
    {
     

In [16]:
# Task 11: Soybean Pod Counting
task = '''
You have a CSV metadata at ./data/soybean_metadata.csv containing 200 records with keys “file_name” (image names), “plant_id”, and “planting_date”.
You task is to extract the number of soybean pods per image.
Merge these results with the existing metadata by adding a new column named pod_count. Save the merged file to ./results/soybean_pod_counts.csv.'''
res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 11
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a CSV metadata at ./data/soybean_metadata.csv containing 200 records with keys “file_name” (image names), “plant_id”, and “planting_date”.
You task is to extract the number of soybean pods per image.
Merge these results with the existing metadata by adding a new column named pod_count. Save the merged file to ./results/soybean_pod_counts.csv.

--------------------------------------------------------------------------------
manager (to Admin):

```json
{
  "tool_uses": [
    {
      "recipient_name": "functions.get_model_zoo",
      "parameters": {}
    },
    {
      "recipient_name": "functions.infer_instance_segmentation",
      "parameters": {
        "file_path": "./data/soybean_metadata.csv",
        "checkpoint": "soybean_pod_segmentation_model",
        "output_dir": 

In [17]:
# Task 12: Training a Drought Stress Identification Model
task = '''
I have some surghum leaf images in ./images/sorghum_leaf/ and metadata in ./data/sorghum_leaf_metadata.csv (columns: file_name, treatment, drought_stress).
I would like to build a model that can automatically identify the drought stress given a surghum leaf image. Can you tell me what to do?'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 12
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
I have some surghum leaf images in ./images/sorghum_leaf/ and metadata in ./data/sorghum_leaf_metadata.csv (columns: file_name, treatment, drought_stress).
I would like to build a model that can automatically identify the drought stress given a surghum leaf image. Can you tell me what to do?

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.get_dataset_format",
        "parameters": {
            "task": "image-classification"
        }
    }
]
```

--------------------------------------------------------------------------------
Admin (to tool_evaluator):

Here is the task provided by the user: 
I have some surghum leaf images in ./images/sorghum_leaf/ and metadata in ./data/sorg

In [18]:
# Task 13: Calculate and Plot Leaf Area/Counting Ratio
task = '''
You have a CSV file ./data/phenotypes.csv with columns: plant_id, leaf_area, leaf_count.
Compute the ratio of leaf_area / leaf_count for each plant and save as ./results/leaf_area_count_ratio.csv. 
Generate a histogram of the ratio values across all plants and save the plot as ./results/leaf_area_ratio_count_hist.png.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 13
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a CSV file ./data/phenotypes.csv with columns: plant_id, leaf_area, leaf_count.
Compute the ratio of leaf_area / leaf_count for each plant and save as ./results/leaf_area_count_ratio.csv. 
Generate a histogram of the ratio values across all plants and save the plot as ./results/leaf_area_ratio_count_hist.png.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.compute_from_csv",
        "parameters": {
            "message": "Compute the ratio of leaf_area / leaf_count for each plant.",
            "file_path": "./data/phenotypes.csv",
            "save_path": "./results/leaf_area_count_ratio.csv"
        }
    },
    {
        "recipient_name": "functions.plot_from_csv",


In [19]:
# Task 14: Rice Growth Stage Summary
task = '''
You have a CSV file at ./data/rice_growth_stages.csv with columns: plant_id, day_after_sowing, growth_stage ("vegetative" or "reproductive").
Plot the number of plants in each growth stage over days after sowing, save the plot at ./results/rice_stage_counts.png, and provide a summary of the plot.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 14
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a CSV file at ./data/rice_growth_stages.csv with columns: plant_id, day_after_sowing, growth_stage ("vegetative" or "reproductive").
Plot the number of plants in each growth stage over days after sowing, save the plot at ./results/rice_stage_counts.png, and provide a summary of the plot.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.plot_from_csv",
        "parameters": {
            "message": "Plot the number of plants in each growth stage over days after sowing.",
            "file_path": "./data/rice_growth_stages.csv",
            "save_path": "./results/rice_stage_counts.png"
        }
    },
    {
        "recipient_name": "functions.analyse_plot",
        "pa

In [20]:
# Task 15: Fruit Size Extraction and Growth Curve Plot
task = '''
You have some fruit images with the metadata at ./data/tomato_metadata.json with keys: file_name, plant_id, day_after_flowering.
Your task is to extract the fruit diameter for each image, save the results at ./results/fruit_diameters.csv.
Also plot the mean fruit diameter over time (day_after_flowering) across all plants. Save the plot at ./results/fruit_growth.png.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 15
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have some fruit images with the metadata at ./data/tomato_metadata.json with keys: file_name, plant_id, day_after_flowering.
Your task is to extract the fruit diameter for each image, save the results at ./results/fruit_diameters.csv.
Also plot the mean fruit diameter over time (day_after_flowering) across all plants. Save the plot at ./results/fruit_growth.png.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.get_model_zoo",
        "parameters": {}
    },
    {
        "recipient_name": "functions.infer_instance_segmentation",
        "parameters": {
            "file_path": "./data/tomato_metadata.json",
            "checkpoint": "your_selected_checkpoint",
            "o

In [22]:
# Task 16: Pipeline Reproduction on Rice Trait Analysis
task = '''
You have a CSV ./data/rice_traits.csv with columns: plant_id, day, leaf_area, treatment.
You need to complete the following tasks:
(1) Compute and plot the mean leaf_area vs. day for each treatment group. Save the plot to ./results/rice_leaf_growth.png.
(2) Save this workflow as a reusable pipeline named rice_trait_analysis.
(3) Apply the pipeline to ./data/rice_traits_new.csv.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 16
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a CSV ./data/rice_traits.csv with columns: plant_id, day, leaf_area, treatment.
You need to complete the following tasks:
(1) Compute and plot the mean leaf_area vs. day for each treatment group. Save the plot to ./results/rice_leaf_growth.png.
(2) Save this workflow as a reusable pipeline named rice_trait_analysis.
(3) Apply the pipeline to ./data/rice_traits_new.csv.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.plot_from_csv",
        "parameters": {
            "message": "Plot the mean of leaf_area against day for each treatment group.",
            "file_path": "./data/rice_traits.csv",
            "save_path": "./results/rice_leaf_growth.png"
        }
    },


In [23]:
# Task 17: Chlorophyll Index Calculation and Plotting
task = '''
Given a CSV file at ./data/leaf_reflectance.csv with columns: plant_id, red, and nir. Red and nir represent reflectance values in the visible red band and near-infrared band, respectively.
Compute the NDVI for each plant using (nir - red) / (nir + red) and merge the results with the original CSV by adding a new column named ndvi. Save the merged file to ./results/leaf_ndvi.csv.
Also plot the NDVI for each plant on a bar chart and save it to ‘./results/ndvi_chart.png’.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 17
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
Given a CSV file at ./data/leaf_reflectance.csv with columns: plant_id, red, and nir. Red and nir represent reflectance values in the visible red band and near-infrared band, respectively.
Compute the NDVI for each plant using (nir - red) / (nir + red) and merge the results with the original CSV by adding a new column named ndvi. Save the merged file to ./results/leaf_ndvi.csv.
Also plot the NDVI for each plant on a bar chart and save it to ‘./results/ndvi_chart.png’.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.compute_from_csv",
        "parameters": {
            "message": "Compute the NDVI using (nir - red) / (nir + red) for each plant and add it as a new column named n

In [24]:
# Task 18: Wheat Biomass Over Treatments
task = '''
You have a CSV ./data/wheat_biomass.csv with columns: plant_id, treatment, biomass.
Compute and plot mean biomass for each treatment group with error bars and save as ./results/wheat_biomass_plot.png.
Analyse the pattern of the plot to determine whether some treatments consistently produce higher biomass than others.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 18
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a CSV ./data/wheat_biomass.csv with columns: plant_id, treatment, biomass.
Compute and plot mean biomass for each treatment group with error bars and save as ./results/wheat_biomass_plot.png.
Analyse the pattern of the plot to determine whether some treatments consistently produce higher biomass than others.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.compute_from_csv",
        "parameters": {
            "message": "Compute the mean and standard error of biomass for each treatment group.",
            "file_path": "./data/wheat_biomass.csv",
            "save_path": "./results/wheat_biomass_summary.csv"
        }
    },
    {
        "recipient_name": "functions.p

In [25]:
# Task 19: Sunflower Head Diameter Growth Trend
task = '''
You have a CSV ./data/sunflower_heads.csv with columns: plant_id, head_diameter, day.
Compute and plot the mean head diameter over days save as ./results/sunflower_head_growth.png. 
Analyse the plotted trend and determine whether head diameter growth appears a linear or an exponential pattern.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 19
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have a CSV ./data/sunflower_heads.csv with columns: plant_id, head_diameter, day.
Compute and plot the mean head diameter over days save as ./results/sunflower_head_growth.png. 
Analyse the plotted trend and determine whether head diameter growth appears a linear or an exponential pattern.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.plot_from_csv",
        "parameters": {
            "message": "Plot the mean head_diameter over days.",
            "file_path": "./data/sunflower_heads.csv",
            "save_path": "./results/sunflower_head_growth.png"
        }
    },
    {
        "recipient_name": "functions.analyse_plot",
        "parameters": {
            "message"

In [26]:
# Task 20: Tomato Ripening Identification
task = '''
You have some tomato fruit images with metadata ./data/tomato_metadata.csv (column: file_name).
Classify each fruit into ripening stages and create a bar chart showing the frequency of each class.
Save the chart as ./results/tomato_ripening_stages.png.'''

res = user_proxy.initiate_chat(recipient=manager, message=prefix+task)

# Evaluate Task 20
eval_message = f'''Here is the task provided by the user: {task}

Here is the chain of tools generated by the agent:
{extract_json_block(res.summary)}'''
eval_res = user_proxy.initiate_chat(recipient=tool_evaluator, message=eval_message)

Admin (to manager):

Output the tools and their arguments in the correct order in JSON format to perform the following task. 
Do not attempt to execute any tools. Do not explain anything.
Here is the task:
You have some tomato fruit images with metadata ./data/tomato_metadata.csv (column: file_name).
Classify each fruit into ripening stages and create a bar chart showing the frequency of each class.
Save the chart as ./results/tomato_ripening_stages.png.

--------------------------------------------------------------------------------
manager (to Admin):

```json
[
    {
        "recipient_name": "functions.get_model_zoo",
        "parameters": {}
    },
    {
        "recipient_name": "functions.infer_image_classification",
        "parameters": {
            "file_path": "./data/tomato_metadata.csv",
            "checkpoint": "your_selected_checkpoint",
            "output_dir": "./results"
        }
    },
    {
        "recipient_name": "functions.plot_from_csv",
        "parameter